[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/weighted_dual/weighted_dual.ipynb)

# Weighted Dual Meshes on the Sphere

This example computes the polar dual of a polyhedron about the unit sphere, and how it moves as the polyhedron's vertices slide along their rays from the centre. Seen from the centre, every vertex has an image on the sphere: the polyhedron's images make a triangle mesh on the sphere, and the images of the dual's vertices make its dual mesh. How far a point lies from the centre does not move its image, but it tilts the faces around it, and with them the dual's vertices: the distances shape the dual mesh, which the mesh on the sphere does not show. The points first breathe in and out along fixed directions; then a tangled dual mesh is untangled by the nearest distances that make every dual edge positive.

Points are antivectors and planes are vectors. A face lies in the plane that pairs to one with each of its vertices, `plane & vertex == 1`, and polarity about the unit sphere is the dual: the face's pole is `plane.dual()`, a point. Whether a pole lies inside its face, and how long a dual edge is, are pairings of planes through the centre with points, each signed against a corner of the mesh, so neither the faces nor space need a hand. Along a fixed ray a point is fixed by its weight, one over its distance, and the folds of the polyhedron and the positions of the poles are linear in the weights.

In the notation of discrete exterior calculus, the dual edge lengths over the primal ones are the diagonal Hodge star on edges, $\star_1$, and the mesh of a hull of points at varying distances is a weighted Delaunay triangulation of their images on the sphere, with its dual mesh the power diagram.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np
from scipy.optimize import LinearConstraint, linprog, minimize

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.surfaces.weighted_dual import render
from examples.surfaces.weighted_dual.core import hull

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
# Points, and directions from the centre, are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()

## 1. Faces and poles

A face's plane is the one that pairs to one with each of its vertices: the planes through the centre and each of its edges, summed, over the volume of its corners, `((a & b) + (b & c) + (c & a)) * (a & b & c).inverse()`. Its dual about the unit sphere, `planes.dual()`, is the face's pole, and the poles are the vertices of the polar dual. On the sphere a pole lands inside its face's image when it lies on the face's side of each of the three planes through the centre and an edge, as the face's third corner does: the two pairings with each such plane, the pole's and the third corner's, agree in sign. The convex hull itself, which triangles the points make, is one step outside the algebra, taken by scipy; the linear programs of the last section are the other.

In [ ]:
# The points, how far their directions are nudged off an even spread, and the seed of the nudges.
n_points = 32
jitter = 0.03
direction_seed = 0

In [ ]:
def directions(count: int) -> Point:
    """Directions spread evenly over the sphere along a Fibonacci spiral, each nudged a little at random."""
    turns = np.arange(count) + 0.5
    polar = np.arccos(1 - 2 * turns / count)
    azimuth = np.pi * (1 + 5**0.5) * turns
    # The pole turned down towards x by its polar angle, then around the pole by its azimuth.
    down = ((mv.z ^ mv.x) * (-polar / 2)).exp()
    around = ((mv.x ^ mv.y) * (-azimuth / 2)).exp()
    spiral = (around * down) >> mv.z.dual()                              # [points] Point
    nudges = mv.vector(np.random.default_rng(direction_seed).normal(size=(count, 3))).dual()   # [points] Point
    return (spiral + jitter * nudges).normalized()


def weighted(directions: Point, weights: np.ndarray) -> Point:
    """Points along the given unit directions at the distances one over their weights, the weights on the
    last axis."""
    return directions * (1 / weights)


def poles(points: Point, faces: np.ndarray) -> Point:
    """The pole of each face with respect to the unit sphere, the dual of the face's plane. The plane is the
    one that pairs to one with each of the face's vertices: the planes through the centre and each edge,
    summed, over the volume of its vertices."""
    a, b, c = (points[..., faces[:, corner]] for corner in range(3))           # [..., faces] Point each
    planes = ((a & b) + (b & c) + (c & a)) * (a & b & c).inverse()             # [..., faces] Plane
    return planes.dual()


def sides(points: Point, faces: np.ndarray, duals: Point) -> Scalar:
    """For each face and each of its edges, positive when the face's dual vertex, seen from the centre, lies
    on the face's side of the plane through the centre and that edge, as the face's third vertex does: all
    three positive when the dual vertex lands inside the face."""
    corners = points[..., faces]                                               # [..., faces, 3] Point
    walls = corners & corners[..., [1, 2, 0]]                                  # [..., faces, 3] Plane
    return (walls & duals[..., None]) * (walls & corners[..., [2, 0, 1]])


def adjacency(faces: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Each edge of a closed triangle mesh, and the two faces either side of it."""
    sides = np.sort(np.stack([faces, np.roll(faces, -1, axis=1)], axis=-1), axis=-1).reshape(-1, 2)
    paired = np.lexsort(sides.T[::-1]).reshape(-1, 2)                          # each edge twice, once per face
    return sides[paired[:, 0]], paired // 3


def opposite(faces: np.ndarray, edges: np.ndarray, face: np.ndarray) -> np.ndarray:
    """The vertex of each given face across from its edge."""
    return faces[face].sum(axis=-1) - edges.sum(axis=-1)


def dual_lengths(points: Point, faces: np.ndarray) -> Scalar:
    """The signed length of each dual edge across its edge, seen from the centre: how much further the pole
    of the face on one side lies to that side of the plane through the centre and the edge than the pole
    of the face on the other side; negative where the two poles have crossed over."""
    edges, beside = adjacency(faces)                                           # [edges, 2] each
    duals = poles(points, faces)                                               # [..., faces] Point
    wall = points[..., edges[:, 0]] & points[..., edges[:, 1]]                 # [..., edges] Plane
    toward = wall & points[..., opposite(faces, edges, beside[:, 1])]          # [..., edges] Scalar
    return (wall & (duals[..., beside[:, 1]] - duals[..., beside[:, 0]])) * toward / toward.norm()


def folds(directions: Point, weights: np.ndarray, faces: np.ndarray) -> Scalar:
    """For each edge of the mesh, how far it bends outward, as on a convex polyhedron, or folds inward where
    negative, for points along the directions at the distances one over their weights: the volume of the
    edge's ends, the third vertex of one face beside it and the far vertex of the other, times all four
    weights. That is a sum over the four corners, each weight times the volume of the other three
    directions, with alternating signs: linear in the weights, the weights on the last axis. Each is scaled
    by the volume of the edge's ends and the first face's third vertex, which sets the sense of the centre."""
    edges, beside = adjacency(faces)                                           # [edges, 2] each
    corners = np.concatenate([edges, opposite(faces, edges, beside[:, 0])[:, None],
                              opposite(faces, edges, beside[:, 1])[:, None]], axis=-1)   # [edges, 4]
    others = corners[:, [[1, 2, 3], [0, 2, 3], [0, 1, 3], [0, 1, 2]]]          # [edges, 4, 3]
    volumes = directions[others[..., 0]] & directions[others[..., 1]] & directions[others[..., 2]]   # [edges, 4] Scalar
    face = volumes[:, 3]                                                       # [edges] Scalar
    return (volumes * (weights[..., corners] * [-1, 1, -1, 1])).sum(axis=-1) * face


def insides(directions: Point, weights: np.ndarray, faces: np.ndarray) -> Scalar:
    """For each face and each of its edges, the signed distance of the face's pole from the plane through the
    centre and that edge, positive on the face's side, for points along the directions at the distances one
    over their weights: all three positive when the pole lands inside the face. The face's plane is the sum
    of the corners' weights times the face's reciprocal frame, the three planes that pair to one with their
    own corner's direction and to zero with the other two, and the planes through the edges are fixed by
    the directions, so this is linear in the weights, the weights on the last axis."""
    a, b, c = (directions[faces[:, corner]] for corner in range(3))           # [faces] Point each
    volume = (a & b & c).inverse()                                             # [faces] Scalar
    frame = stack([(b & c) * volume, (c & a) * volume, (a & b) * volume], axis=-1)   # [faces, 3] Plane
    duals = (frame * weights[..., faces]).sum(axis=-1).dual()                  # [..., faces] Point
    corners = directions[faces]                                                # [faces, 3] Point
    walls = (corners & corners[:, [1, 2, 0]]).normalized()                     # [faces, 3] Plane
    third = walls & corners[:, [2, 0, 1]]                                      # [faces, 3] Scalar
    return (walls & duals[..., None]) * third / third.norm()

## 2. Changing the distances

The points on the sphere keep their mesh there, the spherical Delaunay mesh, throughout: its triangles never change. Each point keeps its direction and moves in and out along it, each with its own phase, so its image on the sphere never moves either. What moves is everything the distances shape: the faces tilt, the polyhedron may stop being convex, and the poles of the faces, the dual vertices, travel across the sphere; once the distances differ the dual is no longer the circumcentric one, and need not even sit well inside its triangles. That freedom, invisible in the mesh on the sphere, is the projective degree of freedom of the dual. The left panel is space: the polyhedron, blue, its vertices coloured by their distance from the sphere, blue inside and red outside, and its polar dual, orange, with the unit sphere faint between them and the parts facing away from the viewer culled. The right panel is the half of the sphere facing the viewer, with the images of both meshes.

In [ ]:
# The seed of the points' phases, their spread in distance from the centre, and the frames of one breath
# and the duration of each.
phase_seed = 3
amplitude = 0.06
n_frames = 48
duration_ms = 80

In [ ]:
def breath() -> Iterator[tuple[Point, np.ndarray, Point]]:
    """At each frame: the points, each at its own distance from the centre along a fixed direction, the
    faces of the mesh they had on the sphere, held throughout, and the poles of those faces."""
    faces = hull(directions(n_points))                                    # [faces, 3]
    phases = np.random.default_rng(phase_seed).uniform(0, 2 * np.pi, n_points)
    radii = 1 + amplitude * np.sin(np.linspace(0, 2 * np.pi, n_frames, endpoint=False)[:, None] + phases)   # [frames, points]
    for points in directions(n_points) * radii:                                # [points] Point
        yield points, faces, poles(points, faces)


frames = list(breath())
well_centred = np.array([(render.scalars(sides(points, faces, duals)) > 0).all(axis=-1).sum() for points, faces, duals in frames])   # [frames]

In [ ]:
print(f"well centred faces over the breath: {well_centred.min()} to {well_centred.max()} of {len(frames[0][1])}")
save_figure(render.draw_complex(*frames[0]), "weighted_dual");

In [ ]:
show_animation(render.animate(frames), "weighted_dual_breath", duration_ms)

## 3. Weights that keep every dual edge positive

The hull of the points at random distances along their rays has a mesh of its own. With every point put back on the sphere the images stay where they were, but the mesh is no longer the sphere's own, and its dual tangles. Its edges cross over: the poles of the two faces beside an edge land on the wrong sides of it, a dual edge of negative length, and those are exactly the edges where the polyhedron folds inward. Negative dual lengths are what break the operators built on a dual, such as a Hodge star with negative entries.

Along a fixed direction a point is fixed by its weight, one over its distance from the centre, and in the weights every edge's fold is linear: it is the volume of the edge's ends and the two vertices across it times their four weights, which is a sum over the four corners of each weight times the volume of the other three directions. So the weights that unfold the whole mesh, every dual edge positive, form a convex cone, and finding them is a linear program, the one other step outside the algebra. The map from weights to folds comes out of the algebra directly, as the folds of one weighting per point, each with that point's weight one and the others zero. The widest margin, the largest smallest fold over all weightings, says whether a rescue exists at all; the rescue shown is the nearest one to the tangled start, the weights whose squared changes sum least while every fold keeps a third of that margin, which shares the change among many points. The pole of a face is linear in the weights too, the corners' weights times the face's reciprocal frame, and the planes through the centre and its edges are fixed, so the pole's signed distance inside each of them is linear as well; so every dual vertex lying inside its triangle is a set of linear conditions too, and joins the folds in the same programs. Since every fold is linear, along the straight line in the weights from the start to the rescue each fold crosses zero at most once: the polyhedron unfolds steadily, edge by edge, and closes in on the rescue ever more slowly over the last quarter of the frames, while the mesh on the sphere stays exactly where it was. The vertices are coloured by their distance from the sphere, blue inside and red outside, each on a stretch of the ray it moves along.

In [ ]:
# The spread and seed of the random distances whose hull gives the mesh the rescue starts from, every point
# on the hull; the bounds on the weights, the share of the widest margin of its conditions the rescue keeps,
# and the frames from the start to the rescue.
mesh_spread, mesh_seed = 0.06, 57
weight_bounds = (0.5, 2.0)
margin_share = 0.3
n_settle_frames = 48

In [ ]:
def fold_map(faces: np.ndarray) -> np.ndarray:
    """The folds of every edge as a linear map of the points' weights, one column per point: the folds with
    that point's weight one and every other zero, all points as one batch."""
    return render.scalars(folds(directions(n_points), np.eye(n_points), faces)).T


def widest(conditions: np.ndarray) -> tuple[np.ndarray, float]:
    """The weights that make the smallest condition, each scaled to unit size, as large as it can be, with
    the weights within their bounds and summing to their count, and that smallest condition: negative when
    no weights meet them all."""
    scaled = conditions / np.linalg.norm(conditions, axis=-1, keepdims=True)
    count = conditions.shape[-1]
    # The weights, then the smallest fold, which is maximised.
    largest = np.concatenate([np.zeros(count), [-1.0]])
    total = np.concatenate([np.ones(count), [0.0]])[None]
    result = linprog(largest, A_ub=np.hstack([-scaled, np.ones((len(scaled), 1))]), b_ub=np.zeros(len(scaled)),
                     A_eq=total, b_eq=[count], bounds=[weight_bounds] * count + [(None, None)])
    return result.x[:-1], result.x[-1]


def inside_map(faces: np.ndarray) -> np.ndarray:
    """Every pole's signed distances inside its face as a linear map of the points' weights, one column per
    point: the distances with that point's weight one and every other zero, all points as one batch."""
    return np.moveaxis(render.scalars(insides(directions(n_points), np.eye(n_points), faces)), 0, -1)


def nearest(conditions: np.ndarray, margin: float, start: np.ndarray) -> np.ndarray:
    """The weights nearest to the start, by the summed squares of the changes, that leave every condition,
    scaled to unit size, at least the given margin: the change shared among many points rather than a few."""
    scaled = conditions / np.linalg.norm(conditions, axis=-1, keepdims=True)
    result = minimize(lambda weights: ((weights - start) ** 2).sum(), start, jac=lambda weights: 2 * (weights - start),
                      method="SLSQP", constraints=[LinearConstraint(scaled, margin, np.inf)], bounds=[weight_bounds] * len(start))
    return result.x


def timing(frames: int) -> np.ndarray:
    """How far along the way each frame is: at a steady pace for three quarters of the frames, then closing in
    on the end over the last quarter, the way left shrinking by so many e-folds, the pace continuous where it
    starts closing in."""
    closing_start, e_folds = 3 / 4, 3
    rate = e_folds / (1 - closing_start)
    left = 1 / (1 + closing_start * rate)                                      # the way left where closing in starts
    time = np.linspace(0, 1, frames)
    return np.where(time < closing_start, time * left * rate, 1 - left * np.exp(-(time - closing_start) * rate))

In [ ]:
radii = np.exp(mesh_spread * np.random.default_rng(mesh_seed).normal(size=n_points))   # [points]
rescued_faces = hull(directions(n_points) * radii)                             # [faces, 3]
start = np.ones(n_points)                                                      # [points]
# Every fold outward, and every pole inside its face.
conditions = np.concatenate([fold_map(rescued_faces), inside_map(rescued_faces).reshape(-1, n_points)])   # [edges + faces * 3, points]
widest_margin = widest(conditions)[1]
rescue = nearest(conditions, margin_share * widest_margin, start)              # [points]
settled = [(points, rescued_faces, poles(points, rescued_faces))
           for points in weighted(directions(n_points), start + timing(n_settle_frames)[:, None] * (rescue - start))]

In [ ]:
for name, (points, faces, duals) in (("at the start", settled[0]), ("rescued", settled[-1])):
    print(f"{name:13s} negative dual edges {(dual_lengths(points, faces).to_array() < 0).sum()} of {len(adjacency(faces)[0])},"
              f" well-centred faces {(sides(points, faces, duals).to_array() > 0).all(axis=-1).sum()} of {len(faces)}")
print(f"widest margin of the conditions {widest_margin:.4f}")
show_animation(render.animate(settled), "weighted_dual_rescue", duration_ms)

In [ ]:
# checks
# On the sphere the dual edges cross over exactly where the polyhedron folds; along the straight line to
# the rescue they only ever uncross, and at the rescue none is left.
crossed = [(dual_lengths(points, faces).to_array() < 0).sum() for points, faces, _ in settled]
assert np.array_equal(dual_lengths(*settled[0][:2]).to_array() < 0,
                      folds(directions(n_points), np.ones(n_points), rescued_faces).to_array() < 0)
assert crossed[0] > 0 and crossed[-1] == 0 and (np.diff(crossed) <= 0).all()
# At the rescue every pole lies inside its face.
assert (sides(*settled[-1]).to_array() > 0).all()
for points, faces, duals in frames:
    # Every face's plane, the dual of its pole, pairs to one with each of the face's vertices.
    assert np.abs((duals.dual_inverse()[:, None] & points[faces]).to_array() - 1).max() < 1e-10


def flip(faces: np.ndarray, edge: int) -> np.ndarray:
    """The mesh with one edge turned to join the other two corners of the faces beside it."""
    edges, beside = adjacency(faces)
    first, second = beside[edge]
    a, b = edges[edge]
    c, d = faces[first].sum() - a - b, faces[second].sum() - a - b
    turned = faces[first][(list(faces[first]).index(a) + 1) % 3] != b
    a, b = (b, a) if turned else (a, b)
    flipped = faces.copy()
    flipped[first], flipped[second] = (a, d, c), (d, b, c)
    return flipped


# A mesh that is no hull at any distances has no rescue, and the same program says so: the sphere's mesh
# with so many random edges flipped, from this seed, has a negative widest margin.
n_flips, flip_seed = 8, 6
rng = np.random.default_rng(flip_seed)
flipped_faces = hull(directions(n_points))                                     # [faces, 3]
for _ in range(n_flips):
    flipped_faces = flip(flipped_faces, rng.integers(len(adjacency(flipped_faces)[0])))
assert widest(fold_map(flipped_faces))[1] < 0